In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

X = df.drop(
    columns=["Churn", "customerID"]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_param_grid = {
    "classifier__n_estimators": [
        200,
        300,
        500,
        700
    ],

    "classifier__max_depth": [
        None,
        5,
        8,
        12,
        16,
        20
    ],

    "classifier__min_samples_split": [
        2,
        5,
        10,
        20
    ],

    "classifier__min_samples_leaf": [
        1,
        2,
        4,
        8
    ],

    "classifier__max_features": [
        "sqrt",
        "log2",
        None
    ],

    "classifier__class_weight": [
        None,
        "balanced"
    ]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

rf_search.fit(
    X_train,
    y_train
)

print("Best parameters:")
print(rf_search.best_params_)

print("Best CV F1:")
print(rf_search.best_score_)

best_rf = rf_search.best_estimator_

rf_results = pd.DataFrame(
    rf_search.cv_results_
)

rf_results[
    [
        "params",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].sort_values(
    "mean_test_score",
    ascending=False
).head(10)

best_rf_cv = rf_results.sort_values(
    "mean_test_score",
    ascending=False
).iloc[0]

print(
    "Train F1:",
    best_rf_cv["mean_train_score"]
)

print(
    "Validation F1:",
    best_rf_cv["mean_test_score"]
)

from xgboost import XGBClassifier
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                random_state=42,
                eval_metric="logloss",
                n_jobs=-1
            )
        )
    ]
)
xgb_param_grid = {
    "classifier__n_estimators": [
        200,
        300,
        500,
        700
    ],

    "classifier__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.1
    ],

    "classifier__max_depth": [
        3,
        4,
        5,
        6,
        8
    ],

    "classifier__min_child_weight": [
        1,
        3,
        5,
        10
    ],

    "classifier__subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "classifier__colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=25,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

xgb_search.fit(
    X_train,
    y_train
)

print("Best XGBoost parameters:")
print(xgb_search.best_params_)

print("Best XGBoost CV F1:")
print(xgb_search.best_score_)

best_xgb = xgb_search.best_estimator_

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            y_prob
        ),
        "PR-AUC": average_precision_score(
            y_test,
            y_prob
        )
    }

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            y_prob
        ),
        "PR-AUC": average_precision_score(
            y_test,
            y_prob
        )
    }   

tuned_results = []

for name, model in {
    "Tuned Random Forest": best_rf,
    "Tuned XGBoost": best_xgb
}.items():

    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )

    metrics["Model"] = name

    tuned_results.append(metrics)

tuned_results_df = pd.DataFrame(
    tuned_results
)

tuned_results_df = tuned_results_df[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC"
    ]
]

tuned_results_df

from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000
            )
        )
    ]
)

logistic_param_grid = {
    "classifier__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "classifier__class_weight": [
        None,
        "balanced"
    ]
}

logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_param_grid,
    n_iter=10,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1
)

logistic_search.fit(
    X_train,
    y_train
)

best_logistic = logistic_search.best_estimator_

print(logistic_search.best_params_)
print(logistic_search.best_score_)

import joblib

joblib.dump(
    best_logistic,
    "../models/logistic_regression_tuned.joblib"
)

joblib.dump(
    best_rf,
    "../models/random_forest_tuned.joblib"
)

joblib.dump(
    best_xgb,
    "../models/xgboost_tuned.joblib"
)


Training shape: (5634, 19)
Testing shape: (1409, 19)


C:\Users\ckaus\AppData\Local\Temp\ipykernel_28136\1413423702.py:48: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters:
{'classifier__n_estimators': 300, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 8, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 20, 'classifier__class_weight': 'balanced'}
Best CV F1:
0.6387420920752052
Train F1: 0.7037429350012314
Validation F1: 0.6387420920752052
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best XGBoost parameters:
{'classifier__subsample': 1.0, 'classifier__n_estimators': 300, 'classifier__min_child_weight': 5, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.03, 'classifier__colsample_bytree': 0.9}
Best XGBoost CV F1:
0.5914222854236959
{'classifier__class_weight': 'balanced', 'classifier__C': 1}
0.6332167190083161


['../models/xgboost_tuned.joblib']